# L4 VAE Anomaly Pipeline (TCP/UDP/ICMP)

Bu notebook sadece **L4** (TCP/UDP/ICMP) anomali detector egitimi icindir.

- Feature engineering
- Train/Val/Test split
- CICIDS2017 normal profil etiketleri (BENIGN) uzerinden VAE egitimi
- Threshold secimi
- Test degerlendirme
- Model + scaler + metadata export

Not: L7 tarafi (HTTP/H2/QUIC davranissal model) bu notebookun disinda, mevcut Mamba pipeline ile ayrik kalir.

In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

In [ ]:
@dataclass
class Config:
    data_glob: str = "data/raw/cicids2017/*.csv"
    output_dir: str = "training/artifacts/l4_vae"
    label_col_candidates: Tuple[str, ...] = ("Label", "label", " Label", "attack", "class")
    protocol_col_candidates: Tuple[str, ...] = ("Protocol", "protocol", "proto")
    normal_profile_labels: Tuple[str, ...] = ("BENIGN", "benign", "normal", "NORMAL")

    # Split: train/val sadece normal profile (BENIGN) icin, test ise benign+attack karmasi
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    # VAE
    hidden_dim: int = 128
    latent_dim: int = 16
    batch_size: int = 1024
    epochs: int = 25
    lr: float = 1e-3
    beta_kl: float = 0.1

    # Threshold
    threshold_quantile: float = 0.995


cfg = Config()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
cfg

In [ ]:
def _find_col(cols: List[str], candidates: Tuple[str, ...]) -> Optional[str]:
    set_cols = {c: c for c in cols}
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in set_cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _map_l4_protocol(v) -> Optional[str]:
    if pd.isna(v):
        return None
    s = str(v).strip().upper()
    # Numeric protocol IDs
    if s in {"6", "TCP"}:
        return "TCP"
    if s in {"17", "UDP"}:
        return "UDP"
    if s in {"1", "ICMP"}:
        return "ICMP"
    return None


def load_l4_dataframe(config: Config) -> pd.DataFrame:
    paths = sorted(Path(".").glob(config.data_glob))
    if not paths:
        raise FileNotFoundError(f"CSV bulunamadi: {config.data_glob}")

    chunks = []
    for p in paths:
        df = pd.read_csv(p, low_memory=False)
        if df.empty:
            continue

        label_col = _find_col(list(df.columns), config.label_col_candidates)
        proto_col = _find_col(list(df.columns), config.protocol_col_candidates)
        if label_col is None or proto_col is None:
            continue

        local = df.copy()
        local["__label_raw"] = local[label_col].astype(str).str.strip()
        local["__is_benign"] = local["__label_raw"].isin(config.normal_profile_labels).astype(int)
        local["__l4_proto"] = local[proto_col].map(_map_l4_protocol)
        local = local[local["__l4_proto"].notna()]
        if not local.empty:
            chunks.append(local)

    if not chunks:
        raise RuntimeError("L4 (TCP/UDP/ICMP) veri bulunamadi. Protokol/etiket kolonlarini kontrol et.")

    out = pd.concat(chunks, ignore_index=True)
    return out


raw_df = load_l4_dataframe(cfg)
print(raw_df.shape)
raw_df[["__l4_proto", "__is_benign"]].value_counts().head(10)

In [ ]:
NON_FEATURE_EXACT = {
    "__label_raw", "__is_benign", "__l4_proto",
    "Label", "label", " Label", "attack", "class",
    "Protocol", "protocol", "proto",
    "Flow ID", "flow id", "Timestamp", "timestamp",
    "Src IP", "Dst IP", "Source IP", "Destination IP",
}


def build_feature_matrix(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    # Sadece numerik kolonlar
    num_df = df.select_dtypes(include=[np.number]).copy()

    # Etiket/yardimci kolonlarin temizligi
    keep_cols = [c for c in num_df.columns if c not in NON_FEATURE_EXACT]
    num_df = num_df[keep_cols]

    # Sabit kolonlar ve yuksek NaN kolonlarini at
    nunique = num_df.nunique(dropna=True)
    variable_cols = nunique[nunique > 1].index.tolist()
    num_df = num_df[variable_cols]

    nan_ratio = num_df.isna().mean()
    good_cols = nan_ratio[nan_ratio < 0.30].index.tolist()
    num_df = num_df[good_cols]

    # NaN/inf temizligi
    num_df = num_df.replace([np.inf, -np.inf], np.nan)
    num_df = num_df.fillna(num_df.median(numeric_only=True))
    num_df = num_df.astype(np.float32)

    return num_df, list(num_df.columns)


X_df, feature_names = build_feature_matrix(raw_df)
y = raw_df["__is_benign"].astype(int).values
proto = raw_df["__l4_proto"].astype(str).values

print("feature_count:", len(feature_names))
print("sample_count:", len(X_df))
X_df.head(2)

In [ ]:
# 1) Tum veri uzerinden test split (stratified benign/anomaly)
idx_all = np.arange(len(X_df))
idx_trainval, idx_test = train_test_split(
    idx_all,
    test_size=cfg.test_ratio,
    random_state=SEED,
    stratify=y,
)

# 2) Train/Val sadece benign profile'dan secilir
trainval_benign = idx_trainval[y[idx_trainval] == 1]
idx_train_benign, idx_val_benign = train_test_split(
    trainval_benign,
    test_size=cfg.val_ratio / (cfg.train_ratio + cfg.val_ratio),
    random_state=SEED,
)

X = X_df.values

scaler = StandardScaler()
X_train = scaler.fit_transform(X[idx_train_benign]).astype(np.float32)
X_val = scaler.transform(X[idx_val_benign]).astype(np.float32)
X_test = scaler.transform(X[idx_test]).astype(np.float32)
y_test = y[idx_test]
proto_test = proto[idx_test]

print("train_benign:", X_train.shape)
print("val_benign:", X_val.shape)
print("test_mixed:", X_test.shape)
print("test benign/anomaly:", int((y_test == 1).sum()), int((y_test == 0).sum()))

In [ ]:
class VAE(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_dim),
        )

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        xr = self.decode(z)
        return xr, mu, logvar


def vae_loss(x, xr, mu, logvar, beta: float = 0.1):
    rec = F.mse_loss(xr, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return rec + beta * kl, rec, kl


def make_loader(arr: np.ndarray, batch_size: int, shuffle: bool):
    t = torch.from_numpy(arr)
    ds = torch.utils.data.TensorDataset(t)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


model = VAE(in_dim=X_train.shape[1], hidden_dim=cfg.hidden_dim, latent_dim=cfg.latent_dim).to(device)
opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

train_loader = make_loader(X_train, cfg.batch_size, True)
val_loader = make_loader(X_val, cfg.batch_size, False)

history = []
best_val = float("inf")
best_state = None

for epoch in range(1, cfg.epochs + 1):
    model.train()
    tr_loss = 0.0
    for (xb,) in train_loader:
        xb = xb.to(device)
        opt.zero_grad()
        xr, mu, logvar = model(xb)
        loss, rec, kl = vae_loss(xb, xr, mu, logvar, beta=cfg.beta_kl)
        loss.backward()
        opt.step()
        tr_loss += float(loss.item()) * xb.size(0)
    tr_loss /= len(train_loader.dataset)

    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for (xb,) in val_loader:
            xb = xb.to(device)
            xr, mu, logvar = model(xb)
            loss, rec, kl = vae_loss(xb, xr, mu, logvar, beta=cfg.beta_kl)
            va_loss += float(loss.item()) * xb.size(0)
    va_loss /= len(val_loader.dataset)

    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss})
    if va_loss < best_val:
        best_val = va_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1 or epoch == cfg.epochs:
        print(f"epoch={epoch:03d} train={tr_loss:.6f} val={va_loss:.6f}")

if best_state is not None:
    model.load_state_dict(best_state)
print("best_val:", best_val)

In [ ]:
def anomaly_score(model: nn.Module, arr: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    model.eval()
    scores = []
    loader = make_loader(arr, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            xr, mu, logvar = model(xb)
            rec_err = torch.mean((xr - xb) ** 2, dim=1)
            kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
            s = rec_err + cfg.beta_kl * kl
            scores.append(s.detach().cpu().numpy())
    return np.concatenate(scores, axis=0)


val_scores = anomaly_score(model, X_val)
threshold = float(np.quantile(val_scores, cfg.threshold_quantile))
print("threshold:", threshold)

test_scores = anomaly_score(model, X_test)
y_pred_anomaly = (test_scores > threshold).astype(int)
y_true_anomaly = (y_test == 0).astype(int)

print(classification_report(y_true_anomaly, y_pred_anomaly, digits=4))
if len(np.unique(y_true_anomaly)) > 1:
    print("roc_auc:", roc_auc_score(y_true_anomaly, test_scores))

for p in ["TCP", "UDP", "ICMP"]:
    m = proto_test == p
    if int(m.sum()) == 0:
        continue
    yt = y_true_anomaly[m]
    yp = y_pred_anomaly[m]
    acc = float((yt == yp).mean())
    print(f"{p}: n={int(m.sum())} acc={acc:.4f}")

In [ ]:
out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

model_path = out_dir / "l4_vae.pt"
scaler_path = out_dir / "l4_scaler.npz"
meta_path = out_dir / "l4_vae_meta.json"
history_path = out_dir / "l4_vae_history.json"

torch.save(model.state_dict(), model_path)
np.savez(
    scaler_path,
    mean_=scaler.mean_.astype(np.float64),
    scale_=scaler.scale_.astype(np.float64),
    var_=scaler.var_.astype(np.float64),
)

meta = {
    "seed": SEED,
    "feature_names": feature_names,
    "input_dim": int(X_train.shape[1]),
    "hidden_dim": int(cfg.hidden_dim),
    "latent_dim": int(cfg.latent_dim),
    "beta_kl": float(cfg.beta_kl),
    "threshold": float(threshold),
    "threshold_quantile": float(cfg.threshold_quantile),
    "protocol_scope": ["TCP", "UDP", "ICMP"],
    "train_samples": int(X_train.shape[0]),
    "val_samples": int(X_val.shape[0]),
    "test_samples": int(X_test.shape[0]),
}

meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

print("saved:")
print(" -", model_path)
print(" -", scaler_path)
print(" -", meta_path)
print(" -", history_path)

## Sonraki Adim (entegrasyon sonraya)

Bu notebooktan cikan `l4_vae.pt + l4_scaler.npz + l4_vae_meta.json` dosyalari daha sonra runtime tarafinda su hook noktasina baglanabilir:

- L4 paket uretimi sonrasi, send oncesi skorlayici katman
- `score <= threshold` ise gonder, degilse mutate/atla

Su an bu notebook bilincli olarak runtime entegrasyonu yapmaz.